В первой части работы был проведён разведочный анализ исходных данных. Мы изучили структуру выборки, состав каналов, длительность записей, технические характеристики сигналов, качество данных и время выполнения таблиц Шульте. Также были выявлены особенности выборки, которые необходимо учитывать при дальнейшем анализе, включая различное количество доступных записей у участников и дисбаланс диагностических групп. Во второй части выполняется подготовка данных непосредственно для машинного обучения. ЭЭГ-записи проходят единую предобработку, из них извлекаются численные признаки, после чего данные объединяются сначала на уровне отдельных записей, а затем на уровне участников. Далее формируются разные наборы признаков и сравниваются модели диагностики ПТСР.


In [5]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import mne
from scipy import signal
from scipy.integrate import trapezoid
#путь к исходным файлам
DATA_PATH = "/home/jupyter/s3/neuro-data"
#целевая частота дискретизации, к которой приводим все данные
SFREQ = 125.0
#каналы, которые смотрим
CHANNELS = ["O1", "T3", "Fp1", "Fp2", "T4", "O2"]
#частотные диапазоны
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}
#самая низкая и высокие частоты, для фильтра
L_FREQ = 1.0
H_FREQ = 40.0
#ЭЭГ анализируется кусками по 4 секунды
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(SFREQ * WINDOW_SEC)
#порог для практически плоского сигнала
FLAT_STD = 1e-12


#максимальная абсолютная амплитуда EEG, MNE возвращает сигнал в вольтах
MAX_ABS_AMP = 300e-6

#максимальный peak-to-peak внутри 4-секундного окна.
MAX_PTP = 500e-6


#гомологичные пары каналов слева/справа
CONNECTIVITY_PAIRS = {
    "Fp1_Fp2": ("Fp1", "Fp2"),
    "T3_T4": ("T3", "T4"),
    "O1_O2": ("O1", "O2"),
}

In [6]:
mne.set_log_level("ERROR")

In [7]:
#смотрим, что находится в конкретном EDF-файле 
def get_condition(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    if "-П" in stem or "-P" in stem:
        return "Rest"
    match = re.search(r"-([1-5])$", stem)
    if match:
        return f"S{match.group(1)}"
    return "Unknown"
#читаем файл, preload=True, загружаем файл в память, так как будем его использовать
def read_edf(path):
    raw = mne.io.read_raw_edf(
        path,
        preload=True,
        verbose='Error'
    )
    #считаем исходную длительность записи
    original_duration = (
        raw.n_times
        /
        raw.info["sfreq"]
    )
    #если запись короче одного 4-секундного окна, дальше ее не обрабатываем
    if original_duration < WINDOW_SEC:
        raise ValueError(
            f"Recording too short: {original_duration:.2f} sec"
        )
    #original_duration = raw.times[-1] #находим длительность
#смотрим, все ли каналы присутствуют
    missing = [ch for ch in CHANNELS if ch not in raw.ch_names]
    if missing:
        raise ValueError(
            f"Missing required channels: {missing}. "
            f"Available channels: {raw.ch_names}"
        )
#отбираем только нужные нам каналы
    raw.pick(CHANNELS)
#читаем исходную частоту, и если нужно, приводим к частоте 125 Гц
    if not np.isclose(raw.info["sfreq"], SFREQ):
        raw.resample(SFREQ, verbose=False)
#оставляем только сигналы с частотами от 1 до 40Гц, это как раз соответствует дельта, тета, альфа и бета ритмам
#phase=zero позволяет избежать сдвига 
    raw.filter(
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        method="fir",
        phase="zero",
        verbose=False
    )
    data = raw.get_data() #превращаем в обычный Numpy массив
    return data, original_duration

**Предобработка ЭЭГ и извлечение признаков**

Каждая доступная EDF-запись обрабатывается одинаковым способом. Из файла выбираются шесть EEG-каналов (O1, T3, Fp1, Fp2, T4, O2), сигнал приводится к общей частоте дискретизации 125 Гц и фильтруется в диапазоне 1–40 Гц. Затем запись делится на четырёхсекундные окна. Для каждого пригодного четырёхсекундного окна методом Welch оценивается спектральная плотность мощности. Для каждого из шести каналов рассчитываются абсолютная логарифмированная и относительная мощности в delta-, theta-, alpha- и beta-диапазонах. Результаты затем агрегируются по окнам. 

In [8]:
def get_psd_features(data):
    #получаем число отсчетов
    n_samples = data.shape[1]
    #создаем новый список, куда будем сохранять признаки каждого временного окна
    window_features = []
    #разбиваем запись на временные окна
    n_windows = n_samples // WINDOW_SAMPLES
    #формируем каждое окно
    for w in range(n_windows):
        start = w * WINDOW_SAMPLES
        stop = start + WINDOW_SAMPLES
        #берем все каналы внутри текущего временного окна
        window = data[:, start:stop]
        #проверяем окно на наличие NaN или бесконечных значений
        if not np.isfinite(window).all():
            continue
        #считаем стандартное отклонение отдельно для каждого канала
        channel_std = np.std(
            window,
            axis=1
        )
        #если хотя бы один канал почти не меняется, исключаем всё окно
        if np.any(channel_std < FLAT_STD):
            continue
        #если хотя бы один канал имеет экстремальную амплитуду, исключаем всё окно
        if np.max(np.abs(window)) > MAX_ABS_AMP:
            continue
        #считаем peak-to-peak отдельно для каждого канала
        peak_to_peak = np.ptp(
            window,
            axis=1
        )
        #исключаем окно, если хотя бы на одном канале размах превышает QC-порог
        if np.any(peak_to_peak > MAX_PTP):
            continue
        #создаем новый словарь признаков для текущего окна
        features = {}
        #проходим по всем ЭЭГ-каналам
        for ch_idx, ch in enumerate(CHANNELS):
            #берем сигнал одного канала
            sig = window[ch_idx]
            #находим спектр мощности, то есть какие частоты представлены в сигнале
            freqs, psd = signal.welch(
                sig,
                fs=SFREQ,
                nperseg=WINDOW_SAMPLES,
                noverlap=0,
                detrend="constant"
            )
            #оставляем только интересующий диапазон 1-40 Гц
            total_mask = (
                (freqs >= L_FREQ)
                &
                (freqs <= H_FREQ)
            )
            #если в этом диапазоне нет точек, канал дальше не анализируем
            if not np.any(total_mask):
                continue
            #считаем общую спектральную мощность в диапазоне 1-40 Гц
            total_power = trapezoid(
                psd[total_mask],
                freqs[total_mask]
            )
            #защита от нулевого значения
            total_power = max(
                total_power,
                1e-12
            )
            #отдельно считаем признаки для каждого частотного диапазона
            for band, (f_low, f_high) in BANDS.items():
                #выбираем частоты текущего диапазона
                mask = (
                    (freqs >= f_low)
                    &
                    (freqs < f_high)
                )

                if not np.any(mask):
                    continue
                #считаем абсолютную мощность внутри выбранного диапазона
                abs_power = trapezoid(
                    psd[mask],
                    freqs[mask]
                )
                #защита от нуля перед логарифмированием
                abs_power = max(
                    abs_power,
                    1e-12
                )
                #считаем долю мощности диапазона от общей мощности 1-40 Гц
                rel_power = (
                    abs_power / total_power
                )
                #сохраняем логарифм абсолютной мощности
                features[
                    f"{ch}_{band}_log"
                ] = np.log(abs_power)
                #сохраняем относительную мощность
                features[
                    f"{ch}_{band}_rel"
                ] = rel_power

        #добавляем окно только если для него рассчитались все признаки
        if len(features) == len(CHANNELS) * len(BANDS) * 2:
            window_features.append(
                features
            )
    #если после QC не осталось ни одного подходящего окна
    if len(window_features) == 0:
        return None, n_windows, 0
    #собираем признаки всех подходящих окон в одну таблицу
    window_df = pd.DataFrame(
        window_features
    )
    #для каждой записи берем медиану признака по всем подходящим окнам
    recording_features = (
        window_df
        .median(axis=0)
        .to_dict()
    )
    return (
        recording_features,
        n_windows,
        len(window_features)
    )

In [9]:
def get_connectivity_features(data):
#рассчитываем функциональную связность между симметричными парами EEG-каналов отдельно для delta, theta, alpha и beta диапазонов
    features = {}
    for pair_name, (ch1, ch2) in CONNECTIVITY_PAIRS.items():
        idx1 = CHANNELS.index(ch1)
        idx2 = CHANNELS.index(ch2)
        freqs, coherence = signal.coherence(
            data[idx1],
            data[idx2],
            fs=SFREQ,
            nperseg=WINDOW_SAMPLES
        )
        for band, (f_low, f_high) in BANDS.items():

            mask = (
                (freqs >= f_low) &
                (freqs <= f_high)
            )

            if not np.any(mask):
                continue
            features[
                f"{pair_name}_{band}_coherence"
            ] = np.nanmean(
                coherence[mask]
            )

    return features

Помимо отдельных EEG-ритмов (дельта, тета, альфа, бета), в спектре есть общий фон: обычно с увеличением частоты мощность сигнала постепенно уменьшается. Эта часть спектра называется апериодической, потому что она не соответствует одному конкретному ритму или частоте. В нашем коде апериодическая составляющая приближённо описывается двумя признаками: exponent характеризует наклон спектра, а offset его общий уровень. Они дополняют обычные спектральные признаки. Так, две записи могут иметь похожую альфа-мощность, но различаться по общей форме спектра.

In [10]:
def get_aperiodic_features(data):
#оцениваем апериодическую часть спектра по линейной зависимости между log10(PSD) и log10(частоты), сохраняем наклон спектра как exponent и свободный член как offset
    features = {}

    for ch_idx, ch in enumerate(CHANNELS):
        freqs, psd = signal.welch(
            data[ch_idx],
            fs=SFREQ,
            nperseg=WINDOW_SAMPLES,
            detrend="constant"
        )
        #не используем частоты ниже 2 Гц,
        #чтобы уменьшить влияние медленного дрейфа
        mask = (
            (freqs >= 2.0) &
            (freqs <= 40.0) &
            np.isfinite(psd) &
            (psd > 0)
        )
        if mask.sum() < 5:
            features[f"{ch}_aperiodic_exponent"] = np.nan
            features[f"{ch}_aperiodic_offset"] = np.nan
            continue

        x = np.log10(
            freqs[mask]
        )

        y = np.log10(
            psd[mask]
        )

        slope, intercept = np.polyfit(
            x,
            y,
            1
        )

        features[
            f"{ch}_aperiodic_exponent"
        ] = -slope

        features[
            f"{ch}_aperiodic_offset"
        ] = intercept

    return features

После определения функций запускается обработка всех доступных EDF-файлов. Для каждой записи выполняются одинаковая предобработка, контроль качества и расчёт ЭЭГ-признаков. Успешные результаты сохраняются в общей таблице, а информация о файлах, которые не удалось обработать, записывается отдельно

In [11]:
#извлекаем спектральные признаки
records = [] #сюда будем добавлять успешно обработанные записи
errors = [] #сюда записи, с которыми возникла проблема
groups = [
    "Норма",
    "ПТСР",
    "Соматоформные"
]

for group in groups:
    group_dir = os.path.join(DATA_PATH, group)
    if not os.path.exists(group_dir):
        print(f"Directory not found: {group_dir}")
        continue
    for subject in sorted(os.listdir(group_dir)):
        subject_dir = os.path.join(group_dir, subject)
        #все что не относится к EDF игнорируется
        if not os.path.isdir(subject_dir):
            continue
        for filename in sorted(os.listdir(subject_dir)):
            if not filename.lower().endswith(".edf"):
                continue
            path = os.path.join(subject_dir, filename)
#превращаем имя файла в Rest, S1-5
            condition = get_condition(filename)
            try:
                data, duration_sec = read_edf(path)
                features, n_windows, n_good_windows = get_psd_features(data) #вынимаем спектральные признаки
                if features is None: #если нет ни одного пригодного окна, добавляем в ошибки и рассчитываем после процент, который попал в ошибки
                    #print(f"No good windows: {filename}")
                    errors.append({
                        "subject_id": subject,
                        "group": group,
                        "file_name": filename,
                        "condition": condition,
                        "error": "No good windows"
                    })
                    continue
                
                #функциональная связность
                connectivity_features = (get_connectivity_features(data))
                #апериодические параметры
                aperiodic_features = (get_aperiodic_features(data))

                #добавляем признаки связности к спектральным признакам
                features.update(connectivity_features)
                #добавляем апериодические признаки
                features.update(aperiodic_features)

                rejected_fraction = (
                    1.0 - n_good_windows / n_windows
                    if n_windows > 0
                    else np.nan
                )
                #создаем строку будущей таблицы из спектральных признаков
                row = features.copy()
#добавляем информацию об испытуемом и записи
                row["subject_id"] = subject
                row["group"] = group
                row["file_name"] = filename
                row["condition"] = condition
                row["task_type"] = (
                    "Rest"
                    if condition == "Rest"
                    else "Schulte"
                    if condition in ["S1", "S2", "S3", "S4", "S5"]
                    else "Unknown"
                )
#добавляем техническую информацию о записи и окнах
                row["duration_sec"] = duration_sec
                row["n_windows"] = n_windows
                row["n_good_windows"] = n_good_windows
                row["rejected_fraction"] = rejected_fraction
                records.append(row)

            except Exception as err:
                error_text = str(err)
                #объединяем похожие ошибки в несколько понятных категорий
                if "Recording too short" in error_text:
                    error_type = (
                        "Recording too short"
                    )
                elif "Bad EDF file" in error_text:
                    error_type = (
                        "Bad EDF file"
                    )
                else:
                    error_type = (
                        "Other"
                    )
                #сохраняем подробную информацию об ошибке
                errors.append({
                    "subject_id": subject,
                    "group": group,
                    "file_name": filename,
                    "condition": condition,
                    "error_type": error_type,
                    "error": error_text
                })

#собираем все успешно обработанные записи в одну таблицу
df = pd.DataFrame(records)
#отдельно собираем таблицу с ошибками
errors_df = pd.DataFrame(errors)
print("Shape:", df.shape)
print("\nГруппы:") #сколько EDF получилось в каждой группе
print(df["group"].value_counts())
print("\nУсловия:") #сколько записей Rest и каждой из пяти таблиц Шульте
print(df["condition"].value_counts())
print("\nВариант записи:") #сколько всего записей покоя и Шульте
print(df["task_type"].value_counts())
#вместо вывода каждой ошибки показываем краткую сводку
print("\n===================================")
print("PROCESSING SUMMARY")
print("===================================")
print(
    "Успешно обработано:",
    len(df)
)
print(
    "Не удалось обработать:",
    len(errors_df)
)
#показываем сколько ошибок каждого типа встретилось
if len(errors_df) > 0:
    print("\nОшибки по типам:")
    print(
        errors_df["error_type"]
        .value_counts()
    )
#подробный список ошибок сохраняем отдельно, чтобы не засорять вывод
errors_df.to_csv(
    "eeg_processing_errors.csv",
    index=False
)


Shape: (971, 81)

Группы:
group
Норма            482
Соматоформные    341
ПТСР             148
Name: count, dtype: int64

Условия:
condition
Rest    211
S1      165
S5      152
S3      149
S4      148
S2      146
Name: count, dtype: int64

Вариант записи:
task_type
Schulte    760
Rest       211
Name: count, dtype: int64

PROCESSING SUMMARY
Успешно обработано: 971
Не удалось обработать: 111

Ошибки по типам:
error_type
Bad EDF file           5
Recording too short    1
Name: count, dtype: int64


Всего было найдено 971 запись: 482 записи нормы, 341 запись соматоформной группы и 148 записей ПТСР. Большую часть составляют записи выполнения таблиц Шульте (760), ещё 211 относятся к состоянию покоя. Количество записей разных условий неодинаково, что говорит о том, что у части участников неполный протокол 

In [12]:
#до этого мы получили спектральные харакетристики для каждого EDF-файла, теперь хотим посмотреть на результат
#отделяем техническую информацию от EEG-признаков
metadata_cols = [
    "subject_id",
    "group",
    "file_name",
    "condition",
    "task_type",
    "duration_sec",
    "n_windows",
    "n_good_windows",
    "rejected_fraction"
]
#все остальные столбцы считаем спектральными ЭЭГ-признаками
feature_cols = [
    c for c in df.columns
    if c not in metadata_cols
]
#ищем записи, где хотя бы один ЭЭГ-признак не рассчитался
bad_rows = df[df[feature_cols].isna().any(axis=1)]
print("\nRows with missing EEG features:", len(bad_rows))
#если такие записи есть, смотрим какой файл 
if len(bad_rows) > 0:
    print(
        bad_rows[
            ["subject_id", "group", "file_name", "condition"]
        ].to_string(index=False)
    )
print("\nЗаписей каждого условия:")
#смотрим, сколько записей каждого условия получилось
print(
    df.groupby(["group", "condition"])
      .size()
      .unstack(fill_value=0)
)
print("\nУчастников:", df["subject_id"].nunique()) #считаем общее количество уникальных испытуемых
print("\nУчастников в группе:") #смотрим количество испытуемых отдельно по группам
print(
    df.groupby("group")["subject_id"]
      .nunique()
)
#сохраняем таблицу со спектральными признаками каждого файла
df.to_csv(
    "eeg_recording_features.csv",
    index=False
)

errors_df.to_csv(
    "eeg_processing_errors.csv",
    index=False
)

print("\nСохранено:")
print("eeg_recording_features.csv")
print("eeg_processing_errors.csv")


Rows with missing EEG features: 0

Записей каждого условия:
condition      Rest  S1  S2  S3  S4  S5
group                                  
Норма           128  87  66  67  67  67
ПТСР             25  24  24  25  25  25
Соматоформные    58  54  56  57  56  60

Участников: 216

Участников в группе:
group
Норма            128
ПТСР              25
Соматоформные     63
Name: subject_id, dtype: int64

Сохранено:
eeg_recording_features.csv
eeg_processing_errors.csv


После обработки формируется таблица отдельных ЭЭГ-записей (recording-level dataset). Одна строка соответствует одной успешно обработанной записи Rest или S1–S5. Дополнительно проверяется отсутствие пропусков среди уже рассчитанных ЭЭГ-признаков и распределение записей по группам и условиям.
В итоговой таблице ЭЭГ-признаков не найжено строк с пропущенными рассчитанными признаками. После обработки обнаружено 216 участников: 128 из группы нормы, 25 с ПТСР и 63 с соматоформными расстройствами. Доступность отдельных условий различается: Rest имеется у 211 участников, тогда как отдельные таблицы Шульте представлены у меньшего числа людей. Это подтверждает, что часть участников прошла неполный протокол. Результаты обработки и информация об ошибках сохраняются отдельно в eeg_recording_features.csv и eeg_processing_errors.csv.

In [14]:
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter(
    "ignore",
    PerformanceWarning
)

**Проверка доступности записей покоя и Шульте**

In [15]:
#основные каналы, диапазоны и условия, с которыми дальше работаем
EEG_CHANNELS = ["O1", "T3", "Fp1", "Fp2", "T4", "O2"]
BANDS = ["delta", "theta", "alpha", "beta"]
CONDITIONS = ["Rest", "S1", "S2", "S3", "S4", "S5"]
#столбцы с информацией о записи
META_COLS = [
    "subject_id",
    "group",
    "file_name",
    "condition",
    "task_type",
    "duration_sec",
    "n_windows",
    "n_good_windows",
    "rejected_fraction"
]
#все остальные столбцы считаем EEG-признаками
FEATURE_COLS = [
    c for c in df.columns
    if c not in META_COLS
]
#проверяем, нет ли у одного человека нескольких файлов одного условия
duplicates = (
    df.groupby(["subject_id", "condition"])
      .size()
      .reset_index(name="n")
)
duplicates = duplicates[duplicates["n"] > 1]
#если дубли есть, выводим их
print("Duplicate subject + condition combinations:",
      len(duplicates))
if len(duplicates) > 0:
    print(duplicates.to_string(index=False))
#строим таблицу наличия Rest и S1-S5 у каждого испытуемого
coverage = (
    df.pivot_table(
        index="subject_id",
        columns="condition",
        values="file_name",
        aggfunc="count"
    )
    .fillna(0)
)
#если какого-то условия вообще нет в данных, добавляем его как нулевой столбец
for condition in CONDITIONS:
    if condition not in coverage.columns:
        coverage[condition] = 0
#оставляем условия в нужном порядке и переводим значения в целые числа
coverage = coverage[CONDITIONS].astype(int)
#смотрим общее количество доступных записей каждого условия
print("\nCondition coverage:")
print(coverage.sum())
#сколько человек имеют полный протокол Rest + S1-S5
print("\nSubjects with all six conditions:",
      ((coverage == 1).all(axis=1)).sum())
#сколько человек имеют запись покоя
print("Subjects with Rest:",
      (coverage["Rest"] == 1).sum())
#сколько человек имеют все пять таблиц Шульте
print("Subjects with all five Schulte trials:",
      ((coverage[["S1", "S2", "S3", "S4", "S5"]] == 1).all(axis=1)).sum())
subject_df = (
    df[["subject_id", "group"]]
    .drop_duplicates()
    .set_index("subject_id")
)
#проверяем, что каждый участник относится только к одной группе
group_counts = (
    df.groupby("subject_id")["group"]
      .nunique()
)
problem_groups = group_counts[group_counts > 1]
if len(problem_groups) > 0:
    print("\nWARNING: subjects with multiple groups:")
    print(problem_groups)
#для каждого человека сохраняем его группу
subject_groups = (
    df.groupby("subject_id")["group"]
      .first()
)
subject_df["group"] = subject_groups
#добавляем ЭЭГ-признаки в покое в таблицу на уровне отдельного испытуемого
rest = (
    df[df["condition"] == "Rest"]
    .set_index("subject_id")
)
for feature in FEATURE_COLS:
    if feature in rest.columns:
        subject_df[f"Rest_{feature}"] = rest[feature]
#добавляем признаки таблицы Шульте
for condition in ["S1", "S2", "S3", "S4", "S5"]:
    trial = (
        df[df["condition"] == condition]
        .set_index("subject_id")
    )
    for feature in FEATURE_COLS:

        if feature in trial.columns:
            subject_df[f"{condition}_{feature}"] = trial[feature]
#считаем средний ЭЭГ-признак по пяти таблицам Шульте, только для людей, у кого есть все 5 таблиц
for feature in FEATURE_COLS:

    trial_columns = [
        f"S1_{feature}",
        f"S2_{feature}",
        f"S3_{feature}",
        f"S4_{feature}",
        f"S5_{feature}"
    ]

    #проверяем, что все нужные столбцы вообще существуют в таблице
    if all(
        c in subject_df.columns
        for c in trial_columns
    ):

        subject_df[
            f"Schulte_mean_{feature}"
        ] = (
            subject_df[trial_columns]
            .mean(
                axis=1,
                skipna=False
            )
        )
#считаем изменение ЭЭГ между покоем и средним по Шульте
for feature in FEATURE_COLS:
    schulte_col = f"Schulte_mean_{feature}"
    rest_col = f"Rest_{feature}"
#считаем разницу только если оба признака существуют
    if schulte_col in subject_df.columns and rest_col in subject_df.columns:

        subject_df[f"Schulte_Rest_{feature}"] = (
            subject_df[schulte_col] - subject_df[rest_col]
        )
#дополнительно считаем разницу между покоем и S1
for feature in FEATURE_COLS:
    rest_col = f"Rest_{feature}"
    s1_col = f"S1_{feature}"
    if (
        rest_col in subject_df.columns
        and s1_col in subject_df.columns
    ):
        subject_df[
            f"S1_Rest_{feature}"
        ] = (
            subject_df[s1_col]
            - subject_df[rest_col]
        )
#считаем изменение ЭЭГ от первой таблицы Шульте к пятой
for feature in FEATURE_COLS:
    s1 = f"S1_{feature}"
    s5 = f"S5_{feature}"
    if s1 in subject_df.columns and s5 in subject_df.columns:

        subject_df[f"S5_S1_{feature}"] = (
            subject_df[s5] - subject_df[s1]
        )
#задаем пары левого и правого каналов
ASYMMETRY_PAIRS = {
    "Fp1_Fp2": ("Fp1", "Fp2"),
    "T3_T4": ("T3", "T4"),
    "O1_O2": ("O1", "O2"),
}
#функция для расчета разницы log-power между левым и правым каналом
def add_asymmetry(prefix):
#проходим по четырем частотным диапазонам
    for band in BANDS:
#проходим по трем парам каналов
        for pair_name, (left, right) in ASYMMETRY_PAIRS.items():
#если оба канала есть, считаем левый минус правый
            left_col = f"{prefix}_{left}_{band}_log"
            right_col = f"{prefix}_{right}_{band}_log"
            if (
                left_col in subject_df.columns
                and right_col in subject_df.columns
            ):
                subject_df[
                    f"{prefix}_{pair_name}_{band}_asym"
                ] = (
                    subject_df[left_col]
                    - subject_df[right_col]
                )
#асимметрия в покое
add_asymmetry("Rest")
#асимметрия первой таблицы Шульте
add_asymmetry("S1")
#изменение лево-правых различий от покоя к S1
add_asymmetry("S1_Rest")
#асимметрия среднего EEG во время Шульте
add_asymmetry("Schulte_mean")
#изменение асимметрии Шульте относительно покоя
add_asymmetry("Schulte_Rest")
#изменение асимметрии от S1 к S5
add_asymmetry("S5_S1")
subject_df = subject_df.copy()

#собираем информацию о наличии каждой попытки Шульте

trial_presence = []

for condition in [
    "S1",
    "S2",
    "S3",
    "S4",
    "S5"
]:
    trial_presence.append(
        coverage[condition] > 0
    )
#считаем количество доступных попыток Шульте
subject_df[
    "n_schulte_trials"
] = (
    pd.concat(
        trial_presence,
        axis=1
    )
    .sum(axis=1)
)
#есть ли запись покоя
subject_df["has_rest"] = (
    coverage["Rest"] > 0
)
#есть ли первая таблица Шульте
subject_df["has_s1"] = (
    coverage["S1"] > 0
)
#есть ли одновременно Rest и S1
subject_df["has_rest_s1"] = (
    (coverage["Rest"] > 0)
    &
    (coverage["S1"] > 0)
)
#есть ли все пять таблиц Шульте
subject_df["has_all_schulte"] = (
    coverage[
        ["S1", "S2", "S3", "S4", "S5"]
    ]
    .gt(0)
    .all(axis=1)
)
#есть ли полный протокол Rest + S1-S5
subject_df["has_full_protocol"] = (
    subject_df["has_rest"]
    &
    subject_df["has_all_schulte"]
)
# ============================================================
#duration_sec для S1-S5 — это фактическое время
#выполнения каждой таблицы Шульте.

behavior = (
    df[
        df["condition"].isin(
            ["S1", "S2", "S3", "S4", "S5"]
        )
    ]
    .pivot_table(
        index="subject_id",
        columns="condition",
        values="duration_sec",
        aggfunc="first"
    )
)
#Добавляем время каждой отдельной таблицы
for i in range(1, 6):

    condition = f"S{i}"

    if condition in behavior.columns:

        subject_df[
            f"behavior_{condition}_time"
        ] = behavior[condition]

behavior_cols = [
    f"behavior_S{i}_time"
    for i in range(1, 6)
]

#среднее время считаем только если доступны все пять таблиц
subject_df[
    "behavior_mean_time"
] = (
    subject_df[behavior_cols]
    .mean(
        axis=1,
        skipna=False
    )
)

#медиану тоже считаем только по полному набору S1-S5
subject_df[
    "behavior_median_time"
] = (
    subject_df[behavior_cols]
    .median(
        axis=1,
        skipna=False
    )
)

#вариабельность времени считаем только по полному набору
subject_df[
    "behavior_std_time"
] = (
    subject_df[behavior_cols]
    .std(
        axis=1,
        skipna=False
    )
)
#изменение времени от первой таблицы к пятой
subject_df[
    "behavior_delta_5_1"
] = (
    subject_df["behavior_S5_time"]
    -
    subject_df["behavior_S1_time"]
)

#считаем общий наклон времени выполнения только по полным пяти попыткам
def calculate_behavior_slope(row):
    trials = []
    times = []

    for i in range(1, 6):

        col = f"behavior_S{i}_time"

        if pd.notna(row[col]):

            trials.append(i)
            times.append(
                row[col]
            )
    #для этого признака требуем полный набор S1-S5
    if len(times) < 5:
        return np.nan
#через имеющиеся точки проводится прямая, нам нужен наклон, показывает насколько в среднем уменьшалось время выполнения с каждой попыткой
    slope = np.polyfit(
        trials,
        times,
        1
    )[0]

    return slope


subject_df[
    "behavior_time_slope"
] = (
    subject_df.apply(
        calculate_behavior_slope,
        axis=1
    )
)

subject_df = subject_df.reset_index()

print("\n===================================")
print("SUBJECT-LEVEL DATASET")
print("===================================")
print("Shape:", subject_df.shape)
#количество людей в каждой группе
print("\nGroups:")
print(subject_df["group"].value_counts())
#сколько людей имеют запись покоя
print("\nSubjects with Rest:")
print(subject_df["has_rest"].value_counts())
#сколько попыток Шульте доступно у разных людей
print("\nSchulte trial availability:")
print(
    subject_df["n_schulte_trials"]
    .value_counts()
    .sort_index()
)
#оставляем только числовые столбцы
numeric_cols = subject_df.select_dtypes(
    include=np.number
).columns
#считаем долю пропусков в каждом числовом признаке
missing_fraction = (
    subject_df[numeric_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)
#смотрим 20 признаков с наибольшим количеством пропусков
print("\nTop 20 features by missingness:")
print(
    missing_fraction.head(20)
)
#сохраняем полную таблицу признаков на уровне испытуемых
subject_df.to_csv(
    "eeg_subject_level_features.csv",
    index=False
)
print("\nSaved:")
print("eeg_subject_level_features.csv")

Duplicate subject + condition combinations: 0

Condition coverage:
condition
Rest    211
S1      165
S2      146
S3      149
S4      148
S5      152
dtype: int64

Subjects with all six conditions: 135
Subjects with Rest: 211
Subjects with all five Schulte trials: 137

SUBJECT-LEVEL DATASET
Shape: (216, 810)

Groups:
group
Норма            128
Соматоформные     63
ПТСР              25
Name: count, dtype: int64

Subjects with Rest:
has_rest
True     211
False      5
Name: count, dtype: int64

Schulte trial availability:
n_schulte_trials
0     42
1     22
2      2
3      3
4     10
5    137
Name: count, dtype: int64

Top 20 features by missingness:
Schulte_Rest_T3_T4_delta_asym      0.375
Schulte_Rest_Fp1_Fp2_delta_asym    0.375
Schulte_Rest_O1_O2_delta_asym      0.375
Schulte_Rest_Fp1_Fp2_theta_asym    0.375
Schulte_Rest_O2_delta_log          0.375
Schulte_Rest_O2_delta_rel          0.375
Schulte_Rest_Fp2_theta_log         0.375
Schulte_Rest_Fp2_theta_rel         0.375
Schulte_Rest_Fp2_a

Из 216 участников Rest доступен для 211, все пять таблиц Шульте — для 137, а полный набор из шести условий — для 135. Следовательно, модель полного протокола может использовать значительно меньше участников, чем модели Rest или Rest+S1

Для машинного обучения необходимо получить одну строку на одного участника. Поэтому признаки отдельных Rest и S1–S5 записей объединяются по subject_id. Дополнительно рассчитываются признаки среднего состояния во время Шульте, изменения относительно покоя, динамика между отдельными пробами, межполушарные асимметрии и поведенческие показатели времени выполнения задания

Далее оставляются участники хотя бы с одной записью Шульте, а индивидуальные признаки S1–S5 после формирования агрегированных характеристик исключаются, чтобы уменьшить размерность и количество структурных пропусков

In [16]:

df = pd.read_csv("eeg_subject_level_features.csv")
print("Original subject-level dataset:")
print("Shape:", df.shape)
print(df["group"].value_counts())

#удаляем записи без таблиц Шульте
df_clean = df[
    df["n_schulte_trials"] > 0
].copy()
print("\n===================================")
print("AFTER SCHULTE FILTER")
print("===================================")
print(
    f"Subjects remaining: {len(df_clean)} "
    f"of {len(df)}"
)
print("\nGroups:")
print(
    df_clean["group"]
    .value_counts()
)

# ============================================================

#удаляем излишние признаки, оставляем: покой, среднее по Шульте, разницу между покоем и Шульте, разницу между s1-s5, признаки ассиметрии
#индивидуальные S1-S5 создают много пропусков.
#они остаются в master dataset, но не входят в baseline ML
#находим признаки отдельных попыток S1-S5
s_specific_cols = [
    c
    for c in df_clean.columns
    if any(
        c.startswith(f"S{i}_")
        for i in range(1, 6)
    )
]
#не удаляем найденную разницу S5-S1
s_specific_cols = [
    c for c in s_specific_cols
    if not c.startswith("S5_S1_")
]
print("\nIndividual S1-S5 columns to remove:")
print(len(s_specific_cols))
df_ml = df_clean.drop(
    columns=s_specific_cols
).copy()

# ============================================================
#проверка на пропуски 
feature_missing = (
    df_ml
    .isna()
    .sum()
    .sort_values(ascending=False)
)
feature_missing = feature_missing[
    feature_missing > 0
]
print("\n===================================")
print("MISSINGNESS")
print("===================================")
print(
    f"Columns with missing values: "
    f"{len(feature_missing)}"
)
if len(feature_missing) > 0:
    print(
        feature_missing.head(30)
    )

#ищем признаки, где нет ни одного значения
empty_features = [
    c
    for c in df_ml.columns
    if df_ml[c].isna().all()
]
print("\nCompletely empty features:")
print(len(empty_features))
#полностью пустые признаки удаляем
if empty_features:
    print(empty_features)
    df_ml = df_ml.drop(
        columns=empty_features
    )
    
#смотрим долю пропусков в признаках, но не удаляем их до кросс-валидации
#пропуски будут заполняться медианой внутри каждого train-fold с помощью SimpleImputer

missing_rate = (
    df_ml
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\n===================================")
print("FEATURE MISSING RATE")
print("===================================")

print(
    missing_rate[
        missing_rate > 0
    ].head(30)
)

#находим все признаки покоя
rest_features = [
    c
    for c in df_ml.columns
    if c.startswith("Rest_")
]
#смотрим сколько Rest-признаков осталось
print("\n===================================")
print("REST FEATURES")
print("===================================")
print(
    "Number of Rest features:",
    len(rest_features)
)
#проверяем у скольких людей не хватает хотя бы одного Rest-признака
if rest_features:
    print(
        "Subjects missing Rest:",
        df_ml[rest_features]
        .isna()
        .any(axis=1)
        .sum()
    )
#по сути проверяем, не получились ли совсем странные числа после всех преобразований
log_features = [
    c
    for c in df_ml.columns
    if c.endswith("_log")
]
rel_features = [
    c
    for c in df_ml.columns
    if c.endswith("_rel")
]
print("\n===================================")
print("EEG FEATURE SANITY CHECK")
print("===================================")
# ============================================================
#смотрим общий диапазон log-power
if log_features:
    print(
        "\nLog-power range:"
    )
    print(
        "min =",
        df_ml[log_features]
        .min()
        .min()
    )
    print(
        "max =",
        df_ml[log_features]
        .max()
        .max()
    )
#смотрим общий диапазон relative-power
if rel_features:
    print(
        "\nRelative-power range:"
    )
    print(
        "min =",
        df_ml[rel_features]
        .min()
        .min()
    )
    print(
        "max =",
        df_ml[rel_features]
        .max()
        .max()
    )

#проверяем relative power на значения вне диапазона от 0 до 1
invalid_rel = (
    (df_ml[rel_features] < 0)
    |
    (df_ml[rel_features] > 1)
)
#считаем, у скольких участников встретилось хотя бы одно такое значение
n_invalid_rel = (
    invalid_rel
    .any(axis=1)
    .sum()
)
print(
    "\nSubjects with invalid relative power:",
    n_invalid_rel
)


# ============================================================
#проверяем, что каждый участник встречается в таблице только один раз
duplicate_subjects = (
    df_ml["subject_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate subjects:",
    duplicate_subjects
)


# ============================================================
#проверяем, что один участник не попал сразу в несколько групп
group_counts = (
    df_ml
    .groupby("subject_id")["group"]
    .nunique()
)
#оставляем только случаи, где у одного subject_id найдено больше одной группы
problem_groups = (
    group_counts[group_counts > 1]
)
print(
    "Subjects with multiple groups:",
    len(problem_groups)
)

# ============================================================
#выводим итоговую информацию о подготовленной таблице
print("\n===================================")
print("FINAL BASELINE ML DATASET")
print("===================================")
print(
    "Shape:",
    df_ml.shape
)
#смотрим сколько участников осталось в каждой группе
print(
    "\nGroups:"
)
print(
    df_ml["group"]
    .value_counts()
)
#смотрим сколько признаков осталось для модели
print(
    "\nFeatures:",
    df_ml.shape[1] - 4
)
#проверяем сколько пропущенных значений осталось во всей таблице
print(
    "\nRemaining missing values:",
    df_ml.isna().sum().sum()
)
#проверяем итоговое количество уникальных участников
print(
    "\nSubjects:",
    df_ml["subject_id"].nunique()
)

# ============================================================
#сохраняем подготовленную таблицу для baseline-модели
df_ml.to_csv(
    "eeg_ml_ready.csv",
    index=False
)
print(
    "\nSaved:"
)
print(
    "eeg_ml_ready.csv"
)

Original subject-level dataset:
Shape: (216, 810)
group
Норма            128
Соматоформные     63
ПТСР              25
Name: count, dtype: int64

AFTER SCHULTE FILTER
Subjects remaining: 174 of 216

Groups:
group
Норма            87
Соматоформные    62
ПТСР             25
Name: count, dtype: int64

Individual S1-S5 columns to remove:
456

MISSINGNESS
Columns with missing values: 346
Schulte_Rest_T4_delta_log         39
Schulte_Rest_Fp2_beta_rel         39
Schulte_Rest_O2_beta_log          39
Schulte_Rest_O2_alpha_rel         39
Schulte_Rest_O2_alpha_log         39
Schulte_Rest_O2_theta_rel         39
Schulte_Rest_O2_theta_log         39
Schulte_Rest_O2_delta_rel         39
Schulte_Rest_O2_delta_log         39
Schulte_Rest_T4_beta_rel          39
Schulte_Rest_O1_beta_log          39
Schulte_Rest_O1_beta_rel          39
Schulte_Rest_O1_alpha_log         39
Schulte_Rest_O1_theta_rel         39
Schulte_Rest_O1_theta_log         39
Schulte_Rest_O1_delta_rel         39
Schulte_Rest_O1_delta_

Отдельно проверяется корректность относительной спектральной мощности. Для исходных Rest/S1/Schulte-признаков relative power должен находиться от 0 до 1. Для признаков-разностей, например S1 − Rest, отрицательные значения являются допустимыми и означают уменьшение относительной мощности. Проверка не выявила некорректных исходных relative-power признаков. Значения Rest и S1 находятся в диапазоне от 0 до 1, а отрицательные значения присутствуют только у признаков изменения между состояниями, что является ожидаемым.

In [17]:
#загружаем полную таблицу на уровне испытуемых
df = pd.read_csv(
    "eeg_subject_level_features.csv"
)


#находим все признаки относительной мощности
rel_cols = [
    c
    for c in df.columns
    if c.endswith("_rel")
]


print(
    "Количество столбцов relative power:",
    len(rel_cols)
)


#отдельно выделяем обычные значения relative power
#для них значения должны находиться в диапазоне от 0 до 1
true_rel_cols = [
    c
    for c in rel_cols
    if not c.startswith("S1_Rest_")
    and not c.startswith("Schulte_Rest_")
    and not c.startswith("S5_S1_")
]


#отдельно выделяем разности relative power между условиями
#они могут быть отрицательными
difference_rel_cols = [
    c
    for c in rel_cols
    if (
        c.startswith("S1_Rest_")
        or c.startswith("Schulte_Rest_")
        or c.startswith("S5_S1_")
    )
]


#смотрим основные характеристики обычных relative power
rel_stats = pd.DataFrame({

    "min":
        df[true_rel_cols].min(),

    "max":
        df[true_rel_cols].max(),

    "mean":
        df[true_rel_cols].mean(),

    "n_negative":
        (df[true_rel_cols] < 0).sum(),

    "n_above_1":
        (df[true_rel_cols] > 1).sum(),

    "missing":
        df[true_rel_cols].isna().sum()

})


#ищем только действительно некорректные значения
invalid_rel_stats = rel_stats[
    (rel_stats["n_negative"] > 0)
    |
    (rel_stats["n_above_1"] > 0)
]


print("\n===================================")
print("ПРОВЕРКА RELATIVE POWER")
print("===================================")

print(
    "Обычных relative-power признаков:",
    len(true_rel_cols)
)

print(
    "Признаков-разностей:",
    len(difference_rel_cols)
)


print(
    "\nСтолбцы с некорректным relative power:"
)

if len(invalid_rel_stats) > 0:

    print(
        invalid_rel_stats
        .sort_values(
            "n_negative",
            ascending=False
        )
    )

else:

    print(
        "Некорректных значений не найдено"
    )


#смотрим общий диапазон обычных relative power
print(
    "\nМинимум обычного relative power:",
    df[true_rel_cols]
    .min()
    .min()
)

print(
    "Максимум обычного relative power:",
    df[true_rel_cols]
    .max()
    .max()
)


#разности проверяем отдельно
#отрицательные значения здесь являются допустимыми
if difference_rel_cols:

    print(
        "\nДиапазон разностей relative power:"
    )

    print(
        "Минимум:",
        df[difference_rel_cols]
        .min()
        .min()
    )

    print(
        "Максимум:",
        df[difference_rel_cols]
        .max()
        .max()
    )


#проверяем relative power отдельно в покое
rest_rel_cols = [
    c
    for c in true_rel_cols
    if c.startswith("Rest_")
]


print("\n===================================")
print("RELATIVE POWER В ПОКОЕ")
print("===================================")

print(
    "Столбцов:",
    len(rest_rel_cols)
)

print(
    "Минимум:",
    df[rest_rel_cols]
    .min()
    .min()
)

print(
    "Максимум:",
    df[rest_rel_cols]
    .max()
    .max()
)

print(
    "Отрицательных значений:",
    (df[rest_rel_cols] < 0)
    .sum()
    .sum()
)

print(
    "Значений больше 1:",
    (df[rest_rel_cols] > 1)
    .sum()
    .sum()
)

#отдельно проверяем relative power первой таблицы Шульте
s1_rel_cols = [
    c
    for c in true_rel_cols
    if c.startswith("S1_")
    and not c.startswith("S1_Rest_")
]


print("\n===================================")
print("RELATIVE POWER В S1")
print("===================================")

print(
    "Столбцов:",
    len(s1_rel_cols)
)

if s1_rel_cols:

    print(
        "Минимум:",
        df[s1_rel_cols]
        .min()
        .min()
    )

    print(
        "Максимум:",
        df[s1_rel_cols]
        .max()
        .max()
    )

    print(
        "Отрицательных значений:",
        (df[s1_rel_cols] < 0)
        .sum()
        .sum()
    )

    print(
        "Значений больше 1:",
        (df[s1_rel_cols] > 1)
        .sum()
        .sum()
    )

#отдельно смотрим изменение relative power от покоя к S1
s1_rest_rel_cols = [
    c
    for c in difference_rel_cols
    if c.startswith("S1_Rest_")
]
print("\n===================================")
print("ИЗМЕНЕНИЕ RELATIVE POWER REST -> S1")
print("===================================")
print(
    "Столбцов:",
    len(s1_rest_rel_cols)
)
if s1_rest_rel_cols:

    print(
        "Минимум:",
        df[s1_rest_rel_cols]
        .min()
        .min()
    )

    print(
        "Максимум:",
        df[s1_rest_rel_cols]
        .max()
        .max()
    )

#загружаем исходные признаки на уровне отдельных EDF
recording_df = pd.read_csv(
    "eeg_recording_features.csv"
)

#на recording-level все _rel являются обычной относительной мощностью
recording_rel_cols = [
    c
    for c in recording_df.columns
    if c.endswith("_rel")
]

print("\n===================================")
print("RECORDING-LEVEL RELATIVE POWER")
print("===================================")
print(
    "Столбцов:",
    len(recording_rel_cols)
)
print(
    "Минимум:",
    recording_df[recording_rel_cols]
    .min()
    .min()
)
print(
    "Максимум:",
    recording_df[recording_rel_cols]
    .max()
    .max()
)
print(
    "Отрицательных значений:",
    (
        recording_df[recording_rel_cols] < 0
    )
    .sum()
    .sum()
)
print(
    "Значений больше 1:",
    (
        recording_df[recording_rel_cols] > 1
    )
    .sum()
    .sum()
)

Количество столбцов relative power: 240

ПРОВЕРКА RELATIVE POWER
Обычных relative-power признаков: 168
Признаков-разностей: 72

Столбцы с некорректным relative power:
Некорректных значений не найдено

Минимум обычного relative power: 0.0081354214986631
Максимум обычного relative power: 0.8676174602998532

Диапазон разностей relative power:
Минимум: -0.6656754910916277
Максимум: 0.5600088384267053

RELATIVE POWER В ПОКОЕ
Столбцов: 24
Минимум: 0.0094419519576871
Максимум: 0.8676174602998532
Отрицательных значений: 0
Значений больше 1: 0

RELATIVE POWER В S1
Столбцов: 24
Минимум: 0.0205235343443797
Максимум: 0.7237951853059739
Отрицательных значений: 0
Значений больше 1: 0

ИЗМЕНЕНИЕ RELATIVE POWER REST -> S1
Столбцов: 24
Минимум: -0.6656754910916277
Максимум: 0.5558523563066256

RECORDING-LEVEL RELATIVE POWER
Столбцов: 24
Минимум: 0.0081354214986631
Максимум: 0.8676174602998532
Отрицательных значений: 0
Значений больше 1: 0


**Основная модель: ПТСР vs контроль**

Основная задача формулируется как бинарная классификация ПТСР - контроль. Для обучения основной модели используются только группы «ПТСР» и «Норма». Соматоформная группа не включается в обучение и далее используется отдельно для проверки специфичности модели. Поэтому далее мы отделяем соматоформную группу

In [18]:
#оставляем для обучения только норму и ПТСР
df_binary = df_ml[
    df_ml["group"].isin(
        ["Норма", "ПТСР"]
    )
].copy()
#создаем целевую переменную для бинарной классификации
#0 - норма
#1 - ПТСР
df_binary["target"] = (
    df_binary["group"] == "ПТСР"
).astype(int)
#смотрим итоговую обучающую выборку
print("===================================")
print("BINARY DATASET")
print("===================================")

print(
    "Shape:",
    df_binary.shape
)
#смотрим количество участников каждой группы
print("\nGroups:")
print(
    df_binary["group"]
    .value_counts()
)
#смотрим распределение целевой переменной
print("\nTarget:")
print(
    df_binary["target"]
    .value_counts()
)
#проверяем количество уникальных испытуемых
print("\nUnique subjects:")
print(
    df_binary["subject_id"]
    .nunique()
)

BINARY DATASET
Shape: (112, 355)

Groups:
group
Норма    87
ПТСР     25
Name: count, dtype: int64

Target:
target
0    87
1    25
Name: count, dtype: int64

Unique subjects:
112


In [19]:
#служебные столбцы не должны использоваться моделью как признаки
NON_FEATURE_COLS = {

    "subject_id",
    "group",
    "target",
    "n_schulte_trials",
    "has_rest",
    "has_s1",
    "has_rest_s1",
    "has_all_schulte",
    "has_full_protocol"
}

#отдельно выделяем поведенческие признаки таблиц Шульте
BEHAVIOR_FEATURES = [
    c
    for c in df_binary.columns
    if c.startswith("behavior_")
]

#все остальные числовые признаки считаем EEG-признаками
EEG_FEATURES = [
    c
    for c in df_binary.columns
    if (
        c not in NON_FEATURE_COLS
        and
        c not in BEHAVIOR_FEATURES
        and
        pd.api.types.is_numeric_dtype(
            df_binary[c]
        )
    )
]
print("===================================")
print("FEATURE SETS")
print("===================================")
#смотрим количество EEG-признаков
print(
    "EEG features:",
    len(EEG_FEATURES)
)
#смотрим количество поведенческих признаков
print(
    "Behavioral features:",
    len(BEHAVIOR_FEATURES)
)
#выводим список поведенческих признаков
print("\nBehavioral features:")

print(
    BEHAVIOR_FEATURES
)

FEATURE SETS
EEG features: 336
Behavioral features: 10

Behavioral features:
['behavior_S1_time', 'behavior_S2_time', 'behavior_S3_time', 'behavior_S4_time', 'behavior_S5_time', 'behavior_mean_time', 'behavior_median_time', 'behavior_std_time', 'behavior_delta_5_1', 'behavior_time_slope']


In [20]:
#загружаем полную таблицу на уровне испытуемых
#здесь признаки отдельных S1-S5 еще не удалены
df_subject = pd.read_csv(
    "eeg_subject_level_features.csv"
)


#оставляем участников, у которых есть и покой, и первая таблица Шульте
df_rest_s1 = df_subject[
    df_subject["has_rest_s1"] == True
].copy()


#для бинарной модели оставляем только норму и ПТСР
#соматоформную группу будем проверять отдельно дальше
df_binary_rest_s1 = df_rest_s1[
    df_rest_s1["group"].isin(
        ["Норма", "ПТСР"]
    )
].copy()


#создаем бинарную целевую переменную
#0 - норма
#1 - ПТСР
df_binary_rest_s1["target"] = (
    df_binary_rest_s1["group"] == "ПТСР"
).astype(int)


#служебные столбцы не должны использоваться как признаки модели
REST_S1_NON_FEATURE_COLS = {
    "subject_id",
    "group",
    "target",
    "n_schulte_trials",
    "has_rest",
    "has_s1",
    "has_rest_s1",
    "has_all_schulte",
    "has_full_protocol"
}


#собираем EEG-признаки покоя
REST_EEG_FEATURES = [
    c
    for c in df_binary_rest_s1.columns
    if (
        c.startswith("Rest_")
        and
        c not in REST_S1_NON_FEATURE_COLS
        and
        pd.api.types.is_numeric_dtype(
            df_binary_rest_s1[c]
        )
    )
]


#собираем EEG-признаки первой таблицы Шульте
S1_EEG_FEATURES = [
    c
    for c in df_binary_rest_s1.columns
    if (
        c.startswith("S1_")
        and
        not c.startswith("S1_Rest_")
        and
        c not in REST_S1_NON_FEATURE_COLS
        and
        pd.api.types.is_numeric_dtype(
            df_binary_rest_s1[c]
        )
    )
]


#собираем изменение EEG от покоя к первой таблице Шульте
S1_REST_EEG_FEATURES = [
    c
    for c in df_binary_rest_s1.columns
    if (
        c.startswith("S1_Rest_")
        and
        pd.api.types.is_numeric_dtype(
            df_binary_rest_s1[c]
        )
    )
]


#объединяем покой, S1 и изменение Rest-S1
REST_S1_EEG_FEATURES = list(
    dict.fromkeys(
        REST_EEG_FEATURES
        +
        S1_EEG_FEATURES
        +
        S1_REST_EEG_FEATURES
    )
)


#из поведенческих признаков здесь используем только время S1
#для него не нужны остальные четыре таблицы Шульте
REST_S1_BEHAVIOR_FEATURES = []

if "behavior_S1_time" in df_binary_rest_s1.columns:
    REST_S1_BEHAVIOR_FEATURES.append(
        "behavior_S1_time"
    )


#смотрим сколько участников вошло в эту выборку
print("\n===================================")
print("BINARY REST + S1 DATASET")
print("===================================")

print(
    "Subjects:",
    len(df_binary_rest_s1)
)

print("\nGroups:")

print(
    df_binary_rest_s1["group"]
    .value_counts()
)


#смотрим сколько признаков получилось
print("\n===================================")
print("REST + S1 FEATURE SETS")
print("===================================")

print(
    "Rest EEG features:",
    len(REST_EEG_FEATURES)
)

print(
    "S1 EEG features:",
    len(S1_EEG_FEATURES)
)

print(
    "Rest-S1 EEG differences:",
    len(S1_REST_EEG_FEATURES)
)

print(
    "All Rest + S1 EEG features:",
    len(REST_S1_EEG_FEATURES)
)

print(
    "Rest + S1 behavioral features:",
    REST_S1_BEHAVIOR_FEATURES
)


BINARY REST + S1 DATASET
Subjects: 111

Groups:
group
Норма    87
ПТСР     24
Name: count, dtype: int64

REST + S1 FEATURE SETS
Rest EEG features: 84
S1 EEG features: 84
Rest-S1 EEG differences: 84
All Rest + S1 EEG features: 252
Rest + S1 behavioral features: ['behavior_S1_time']


In [21]:
from sklearn.model_selection import (
    LeaveOneGroupOut,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    SelectKBest,
    f_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix
)

Для всех вариантов модели используется одна схема оценки — nested LOSO. На каждом внешнем шаге один участник полностью оставляется для тестирования, а модель обучается на остальных. Внутри обучающей части выполняются заполнение пропусков, стандартизация, отбор признаков и подбор параметров логистической регрессии

In [22]:
def run_nested_loso_binary(
    data,
    feature_list,
    model_name
):
    """
    Запускаем LOSO на уровне испытуемых.
    Внутри каждого train-fold подбираем гиперпараметры
    только по обучающим данным.
    """

    X = data[
        feature_list
    ].copy()

    y = data[
        "target"
    ].copy()

    groups = data[
        "subject_id"
    ].copy()

    # внешняя CV:
    # каждый раз один испытуемый остается тестовым
    outer_cv = LeaveOneGroupOut()

    y_true = []
    y_probability = []
    y_pred = []
    subject_ids = []
    best_params = []

    for train_idx, test_idx in outer_cv.split(
        X,
        y,
        groups
    ):

        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]

        y_train = y.iloc[
            train_idx
        ]

        y_test = y.iloc[
            test_idx
        ]

        pipeline = Pipeline([

            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            ),

            (
                "selector",
                SelectKBest(
                    score_func=f_classif
                )
            ),

            (
                "classifier",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=42
                )
            )
        ])

        possible_k = [
            k
            for k in [10, 20, 30, 50]
            if k <= len(feature_list)
        ]

        possible_k.append(
            "all"
        )

        parameter_grid = {

            "selector__k":
                possible_k,

            "classifier__C":
                [
                    0.01,
                    0.1,
                    1.0,
                    10.0
                ]
        }

        inner_cv = StratifiedKFold(
            n_splits=4,
            shuffle=True,
            random_state=42
        )

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=parameter_grid,
            scoring="roc_auc",
            cv=inner_cv,
            n_jobs=-1,
            refit=True
        )

        search.fit(
            X_train,
            y_train
        )

        probability = (
            search.predict_proba(
                X_test
            )[:, 1]
        )

        prediction = (
            probability >= 0.5
        ).astype(int)

        y_true.extend(
            y_test.tolist()
        )

        y_probability.extend(
            probability.tolist()
        )

        y_pred.extend(
            prediction.tolist()
        )

        subject_ids.extend(
            groups.iloc[
                test_idx
            ].tolist()
        )

        best_params.append(
            search.best_params_
        )

    auc = roc_auc_score(
        y_true,
        y_probability
    )

    balanced_acc = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    predictions = pd.DataFrame({

        "subject_id":
            subject_ids,

        "true_target":
            y_true,

        "ptsd_probability":
            y_probability,

        "predicted_target":
            y_pred
    })

    return {

        "model_name":
            model_name,

        "auc":
            auc,

        "balanced_accuracy":
            balanced_acc,

        "accuracy":
            accuracy,

        "confusion_matrix":
            cm,

        "predictions":
            predictions,

        "best_params":
            best_params
    }

**Модель на основе записи покоя**

Сначала оценивается базовый вариант, использующий только EEG покоя. Он показывает, насколько хорошо Норма и ПТСР различаются без информации о выполнении таблиц Шульте

In [23]:
#обучаем модель только по EEG покоя
#используем людей, у которых есть и Rest, и S1,
#чтобы потом честно сравнить результат с моделью Rest + S1
result_rest = run_nested_loso_binary(
    data=df_binary_rest_s1,
    feature_list=REST_EEG_FEATURES,
    model_name="Rest EEG"
)
print("===================================")
print("REST EEG")
print("===================================")
#выводим ROC-AUC
print(
    "ROC-AUC:",
    round(
        result_rest["auc"],
        3
    )
)
#выводим balanced accuracy
print(
    "Balanced accuracy:",
    round(
        result_rest[
            "balanced_accuracy"
        ],
        3
    )
)


#выводим обычную accuracy
print(
    "Accuracy:",
    round(
        result_rest["accuracy"],
        3
    )
)


#выводим матрицу ошибок
print(
    "\nConfusion matrix:"
)

print(
    result_rest[
        "confusion_matrix"
    ]
)

REST EEG
ROC-AUC: 0.847
Balanced accuracy: 0.79
Accuracy: 0.766

Confusion matrix:
[[65 22]
 [ 4 20]]


**Модель на основе изменений Rest-S1**

Для эксперимента Rest+S1 используется исходная subject-level таблица, поскольку индивидуальные S1-признаки были удалены из базовой ML-ветки. Оставляются участники, имеющие одновременно Rest и S1, после чего формируется набор из признаков покоя, первой таблицы Шульте и изменения EEG между S1 и Rest

In [24]:
#обучаем модель по EEG покоя,
#первой таблице Шульте и изменению Rest-S1
result_rest_s1 = run_nested_loso_binary(
    data=df_binary_rest_s1,
    feature_list=REST_S1_EEG_FEATURES,
    model_name="Rest + S1 EEG"
)


print("\n===================================")
print("REST + S1 EEG")
print("===================================")


#выводим ROC-AUC
print(
    "ROC-AUC:",
    round(
        result_rest_s1["auc"],
        3
    )
)


#выводим balanced accuracy
print(
    "Balanced accuracy:",
    round(
        result_rest_s1[
            "balanced_accuracy"
        ],
        3
    )
)


#выводим обычную accuracy
print(
    "Accuracy:",
    round(
        result_rest_s1["accuracy"],
        3
    )
)


#выводим матрицу ошибок
print(
    "\nConfusion matrix:"
)

print(
    result_rest_s1[
        "confusion_matrix"
    ]
)


REST + S1 EEG
ROC-AUC: 0.759
Balanced accuracy: 0.722
Accuracy: 0.73

Confusion matrix:
[[64 23]
 [ 7 17]]


In [25]:
#добавляем к EEG время выполнения первой таблицы Шульте
#остальные поведенческие признаки здесь не используем,
#потому что они требуют S2-S5
result_rest_s1_behavior = (
    run_nested_loso_binary(
        data=df_binary_rest_s1,
        feature_list=(
            REST_S1_EEG_FEATURES
            +
            REST_S1_BEHAVIOR_FEATURES
        ),
        model_name="Rest + S1 EEG + behavior"
    )
)


print("\n===================================")
print("REST + S1 EEG + BEHAVIOR")
print("===================================")


#выводим ROC-AUC
print(
    "ROC-AUC:",
    round(
        result_rest_s1_behavior[
            "auc"
        ],
        3
    )
)


#выводим balanced accuracy
print(
    "Balanced accuracy:",
    round(
        result_rest_s1_behavior[
            "balanced_accuracy"
        ],
        3
    )
)


#выводим обычную accuracy
print(
    "Accuracy:",
    round(
        result_rest_s1_behavior[
            "accuracy"
        ],
        3
    )
)


#выводим матрицу ошибок
print(
    "\nConfusion matrix:"
)

print(
    result_rest_s1_behavior[
        "confusion_matrix"
    ]
)


REST + S1 EEG + BEHAVIOR
ROC-AUC: 0.742
Balanced accuracy: 0.749
Accuracy: 0.748

Confusion matrix:
[[65 22]
 [ 6 18]]


**Модель на основе полного протокола**

Для модели полного протокола оставляются только участники с Rest и всеми пятью таблицами Шульте. Модель использует признаки покоя, средние характеристики EEG во время Шульте и показатели изменения EEG между состояниями

In [27]:
#возвращаемся к основной бинарной выборке
#и оставляем только участников с полным протоколом

df_binary_full = df_binary[
    (df_binary["n_schulte_trials"] == 5)
    &
    (df_binary["has_rest"] == True)
].copy()


#собираем EEG-признаки полного протокола
FULL_EEG_FEATURES = [
    c
    for c in EEG_FEATURES
    if (
        c.startswith("Rest_")
        or c.startswith("Schulte_mean_")
        or c.startswith("Schulte_Rest_")
        or c.startswith("S5_S1_")
    )
]


#проверяем выборку перед обучением
print("===================================")
print("FULL PROTOCOL DATASET")
print("===================================")

print(
    "Subjects:",
    len(df_binary_full)
)

print("\nGroups:")

print(
    df_binary_full["group"]
    .value_counts()
)

print(
    "\nFull EEG features:",
    len(FULL_EEG_FEATURES)
)

FULL PROTOCOL DATASET
Subjects: 90

Groups:
group
Норма    66
ПТСР     24
Name: count, dtype: int64

Full EEG features: 336


In [28]:
#обучаем модель по EEG полного протокола
#используем только людей с Rest и всеми пятью таблицами Шульте
result_full_eeg = run_nested_loso_binary(
    data=df_binary_full,
    feature_list=FULL_EEG_FEATURES,
    model_name="Full EEG"
)


print("\n===================================")
print("FULL EEG")
print("===================================")


#выводим ROC-AUC
print(
    "ROC-AUC:",
    round(
        result_full_eeg["auc"],
        3
    )
)


#выводим balanced accuracy
print(
    "Balanced accuracy:",
    round(
        result_full_eeg[
            "balanced_accuracy"
        ],
        3
    )
)


#выводим обычную accuracy
print(
    "Accuracy:",
    round(
        result_full_eeg["accuracy"],
        3
    )
)


#выводим матрицу ошибок
print(
    "\nConfusion matrix:"
)

print(
    result_full_eeg[
        "confusion_matrix"
    ]
)


FULL EEG
ROC-AUC: 0.872
Balanced accuracy: 0.816
Accuracy: 0.789

Confusion matrix:
[[50 16]
 [ 3 21]]


**Модель на основе полного протокола и поведенческих характеристик**

К полному EEG-набору добавляются показатели времени выполнения таблиц Шульте. Этот эксперимент позволяет проверить, улучшает ли поведенческая информация классификацию по сравнению с использованием только EEG

In [29]:
#добавляем к EEG полного протокола
#все поведенческие признаки таблиц Шульте
result_full_eeg_behavior = (
    run_nested_loso_binary(
        data=df_binary_full,
        feature_list=(
            FULL_EEG_FEATURES
            +
            BEHAVIOR_FEATURES
        ),
        model_name="Full EEG + behavior"
    )
)
print("\n===================================")
print("FULL EEG + BEHAVIOR")
print("===================================")
#выводим ROC-AUC
print(
    "ROC-AUC:",
    round(
        result_full_eeg_behavior[
            "auc"
        ],
        3
    )
)
#выводим balanced accuracy
print(
    "Balanced accuracy:",
    round(
        result_full_eeg_behavior[
            "balanced_accuracy"
        ],
        3
    )
)
#выводим обычную accuracy
print(
    "Accuracy:",
    round(
        result_full_eeg_behavior[
            "accuracy"
        ],
        3
    )
)
#выводим матрицу ошибок
print(
    "\nConfusion matrix:"
)
print(
    result_full_eeg_behavior[
        "confusion_matrix"
    ]
)


FULL EEG + BEHAVIOR
ROC-AUC: 0.918
Balanced accuracy: 0.86
Accuracy: 0.833

Confusion matrix:
[[53 13]
 [ 2 22]]


In [30]:
#сравниваем две модели полного протокола
comparison_full = pd.DataFrame([

    {
        "model":
            "Full EEG",

        "ROC_AUC":
            result_full_eeg["auc"],

        "balanced_accuracy":
            result_full_eeg[
                "balanced_accuracy"
            ],

        "accuracy":
            result_full_eeg["accuracy"]
    },

    {
        "model":
            "Full EEG + behavior",

        "ROC_AUC":
            result_full_eeg_behavior[
                "auc"
            ],

        "balanced_accuracy":
            result_full_eeg_behavior[
                "balanced_accuracy"
            ],

        "accuracy":
            result_full_eeg_behavior[
                "accuracy"
            ]
    }
])


#выводим сравнение
display(
    comparison_full.round(3)
)

,model,ROC_AUC,balanced_accuracy,accuracy
0,Full EEG,0.872,0.816,0.789
1,Full EEG + behavior,0.918,0.860,0.833


**Сравнение моделей на одинаковых участниках**

Первоначально разные модели могли использовать разное число участников из-за неполного протокола. Поэтому для корректного сравнения Rest, Rest+S1 и полного протокола формируется одна и та же выборка участников. Это позволяет связать различия метрик именно с набором признаков, а не с изменением состава выборки

In [31]:
#повторно проверяем Rest только на участниках полного протокола
#так Rest и Full EEG будут оцениваться на одних и тех же людях
result_rest_full_sample = run_nested_loso_binary(
    data=df_binary_full,
    feature_list=REST_EEG_FEATURES,
    model_name="Rest EEG - full protocol sample"
)


print("\n===================================")
print("REST EEG - FULL PROTOCOL SAMPLE")
print("===================================")


print(
    "ROC-AUC:",
    round(
        result_rest_full_sample["auc"],
        3
    )
)


print(
    "Balanced accuracy:",
    round(
        result_rest_full_sample[
            "balanced_accuracy"
        ],
        3
    )
)


print(
    "Accuracy:",
    round(
        result_rest_full_sample[
            "accuracy"
        ],
        3
    )
)


print(
    "\nConfusion matrix:"
)

print(
    result_rest_full_sample[
        "confusion_matrix"
    ]
)


REST EEG - FULL PROTOCOL SAMPLE
ROC-AUC: 0.868
Balanced accuracy: 0.795
Accuracy: 0.778

Confusion matrix:
[[50 16]
 [ 4 20]]


In [32]:
#берем тех же участников, которые вошли в полный протокол
full_subject_ids = set(
    df_binary_full["subject_id"]
)


#из Rest-S1 выборки оставляем только этих же людей
df_binary_rest_s1_full_sample = (
    df_binary_rest_s1[
        df_binary_rest_s1[
            "subject_id"
        ].isin(
            full_subject_ids
        )
    ]
    .copy()
)


print(
    "Subjects:",
    len(
        df_binary_rest_s1_full_sample
    )
)

print(
    df_binary_rest_s1_full_sample[
        "group"
    ].value_counts()
)

Subjects: 90
group
Норма    66
ПТСР     24
Name: count, dtype: int64


In [33]:
#проверяем Rest + S1 на той же выборке,
#что и полный протокол
result_rest_s1_full_sample = (
    run_nested_loso_binary(
        data=df_binary_rest_s1_full_sample,
        feature_list=REST_S1_EEG_FEATURES,
        model_name="Rest + S1 EEG - full protocol sample"
    )
)

In [34]:
print("\n===================================")
print("REST + S1 EEG - FULL PROTOCOL SAMPLE")
print("===================================")


print(
    "ROC-AUC:",
    round(
        result_rest_s1_full_sample["auc"],
        3
    )
)


print(
    "Balanced accuracy:",
    round(
        result_rest_s1_full_sample[
            "balanced_accuracy"
        ],
        3
    )
)


print(
    "Accuracy:",
    round(
        result_rest_s1_full_sample[
            "accuracy"
        ],
        3
    )
)


print(
    "\nConfusion matrix:"
)

print(
    result_rest_s1_full_sample[
        "confusion_matrix"
    ]
)


REST + S1 EEG - FULL PROTOCOL SAMPLE
ROC-AUC: 0.938
Balanced accuracy: 0.919
Accuracy: 0.9

Confusion matrix:
[[58  8]
 [ 1 23]]


Rest+S1 показала заметно разные результаты на широкой выборке и на выборке полного протокола. Поэтому отдельно проверяется, какие именно участники были исключены при переходе к общей выборке. Это необходимо, чтобы понять, связано ли изменение ROC-AUC с составом контрольной группы.
 Анализ показал, что 20 из 21 исключённого участника нормы имеют возраст 65–70 лет и только одну Schulte-запись. Поэтому они доступны для моделей Rest и Rest+S1, но не могут быть корректно использованы для моделей полного протокола S1–S5. Одновременно это показывает, что рост ROC-AUC Rest+S1 с 0.759 до 0.938 сопровождался исключением отдельной возрастной части контрольной выборки. Поэтому специфичность модели на здоровых пожилых участниках необходимо оценивать отдельно

**Выбор модели**

Для решения бинарной задачи различения ПТСР и нормы было проверено несколько вариантов модели, отличавшихся объёмом используемой информации. Базовая модель использовала только ЭЭГ покоя и на общей сопоставимой выборке показала ROC-AUC около 0.86. Добавление ЭЭГ первой таблицы Шульте и признаков изменения между Rest и S1 повысило наблюдаемый ROC-AUC до 0.938. Использование полного ЭЭГ-протокола Rest + S1–S5 не привело к дальнейшему улучшению: ROC-AUC составил 0.872. При добавлении к полному EEG-протоколу времени выполнения таблиц Шульте ROC-AUC увеличился до 0.918.
Для корректного сравнения эти результаты были получены на одинаковой выборке из 90 участников — 66 человек основной контрольной группы и 24 участников с ПТСР. Наиболее высокие наблюдаемые показатели получила модель Rest+S1 ЭЭГ со значением ROC-AUC 0.938. При этом модель требует только двух ЭЭГ-записей, тогда как варианты полного протокола требуют наличия всех пяти таблиц Шульте.
В ходе анализа была обнаружена дополнительная особенность выборки. При первоначальной оценке Rest+S1 использовалась более широкая выборка из 111 человек, 87 Норма и 24 ПТСР, при этом, ROC-AUC составлял только 0.759. При переходе к общей выборке полного протокола были исключены 21 участник нормы. Двадцать из них оказались здоровыми участниками возраста 65–70 лет, имеющими Rest и только одну запись Шульте. После их исключения ROC-AUC Rest+S1 увеличился до 0.938.
Таким образом, изменение результата связано не с изменением алгоритма Rest+S1, а с изменением состава контрольной выборки. Это является важным ограничением, так как высокая эффективность Rest+S1 на основной выборке ещё не гарантирует хорошее различение ПТСР и здоровых людей старшего возраста. Rest+S1 EEG была выбырана в качестве финальной модели благодаря высоким показателям на основной выборке и возможностям применения к участникам с неполным протоколом. Однако её потенциальным недостатком является возможное снижение специфичности на здоровой возрастной группе, что требует отдельной проверки перед окончательными выводами.

In [4]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

ROOT = Path("/home/jupyter/project")

DATA_FILE = ROOT / "eeg_subject_level_features.csv"
MODEL_DIR = ROOT / "model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "final_model.joblib"
FEATURES_PATH = MODEL_DIR / "feature_names.json"
METADATA_PATH = MODEL_DIR / "metadata.json"


df_subject = pd.read_csv(DATA_FILE)

print("Loaded:", DATA_FILE)
print("Shape:", df_subject.shape)



#Участники, у которых есть и Rest, и S1
df_rest_s1 = df_subject[
    df_subject["has_rest_s1"] == True
].copy()

#Только Норма и ПТСР
df_binary_rest_s1 = df_rest_s1[
    df_rest_s1["group"].isin(["Норма", "ПТСР"])
].copy()

# 0 = Норма
# 1 = ПТСР
df_binary_rest_s1["target"] = (
    df_binary_rest_s1["group"] == "ПТСР"
).astype(int)


full_protocol = df_binary_rest_s1[
    df_binary_rest_s1["n_schulte_trials"] == 5
].copy()

full_subject_ids = set(
    full_protocol["subject_id"]
)

df_final = df_binary_rest_s1[
    df_binary_rest_s1["subject_id"].isin(full_subject_ids)
].copy()


print("\nFinal dataset:")
print("Subjects:", df_final["subject_id"].nunique())
print("\nGroups:")
print(df_final["group"].value_counts())


NON_FEATURE_COLS = {
    "subject_id",
    "group",
    "target",
    "n_schulte_trials",
    "has_rest",
    "has_s1",
    "has_rest_s1",
    "has_all_schulte",
    "has_full_protocol",
}


# Rest EEG
REST_EEG_FEATURES = [
    c
    for c in df_final.columns
    if (
        c.startswith("Rest_")
        and c not in NON_FEATURE_COLS
        and pd.api.types.is_numeric_dtype(df_final[c])
    )
]


# S1 EEG
S1_EEG_FEATURES = [
    c
    for c in df_final.columns
    if (
        c.startswith("S1_")
        and not c.startswith("S1_Rest_")
        and c not in NON_FEATURE_COLS
        and pd.api.types.is_numeric_dtype(df_final[c])
    )
]


# S1 - Rest
S1_REST_EEG_FEATURES = [
    c
    for c in df_final.columns
    if (
        c.startswith("S1_Rest_")
        and pd.api.types.is_numeric_dtype(df_final[c])
    )
]



REST_S1_EEG_FEATURES = list(
    dict.fromkeys(
        REST_EEG_FEATURES
        + S1_EEG_FEATURES
        + S1_REST_EEG_FEATURES
    )
)


print("\nFeature counts:")
print("Rest:", len(REST_EEG_FEATURES))
print("S1:", len(S1_EEG_FEATURES))
print("S1-Rest:", len(S1_REST_EEG_FEATURES))
print("Total:", len(REST_S1_EEG_FEATURES))


X = df_final[REST_S1_EEG_FEATURES].copy()
y = df_final["target"].copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)



pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "selector",
        SelectKBest(score_func=f_classif)
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        )
    )
])


possible_k = [
    k
    for k in [10, 20, 30, 50]
    if k <= len(REST_S1_EEG_FEATURES)
]

possible_k.append("all")


parameter_grid = {
    "selector__k": possible_k,
    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0
    ]
}



inner_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)


search_final = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1,
    refit=True
)

print("\nTraining final model...")

search_final.fit(X, y)


final_model = search_final.best_estimator_

joblib.dump(
    final_model,
    MODEL_PATH
)

with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        REST_S1_EEG_FEATURES,
        f,
        ensure_ascii=False,
        indent=2
    )


metadata = {
    "model_type": "LogisticRegression",
    "feature_set": "Rest + S1 EEG",
    "n_subjects": int(df_final["subject_id"].nunique()),
    "n_normal": int((y == 0).sum()),
    "n_ptsd": int((y == 1).sum()),
    "n_features": int(len(REST_S1_EEG_FEATURES)),
    "best_params": search_final.best_params_,
    "cv_scoring": "roc_auc",
    "inner_cv": "StratifiedKFold(n_splits=4, shuffle=True, random_state=42)",
    "threshold": 0.5
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )



print("Model:", MODEL_PATH)
print("Features:", FEATURES_PATH)
print("Metadata:", METADATA_PATH)

print("\nBest parameters:")
print(search_final.best_params_)

print("\nBest inner-CV ROC-AUC:")
print(round(search_final.best_score_, 3))

print("\nModel:")
print(final_model)

Loaded: /home/jupyter/project/eeg_subject_level_features.csv
Shape: (216, 810)

Final dataset:
Subjects: 90

Groups:
group
Норма    66
ПТСР     24
Name: count, dtype: int64

Feature counts:
Rest: 84
S1: 84
S1-Rest: 84
Total: 252

X shape: (90, 252)
y shape: (90,)

Training final model...
Model: /home/jupyter/project/model/final_model.joblib
Features: /home/jupyter/project/model/feature_names.json
Metadata: /home/jupyter/project/model/metadata.json

Best parameters:
{'classifier__C': 1.0, 'selector__k': 'all'}

Best inner-CV ROC-AUC:
0.943

Model:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('selector', SelectKBest(k='all')),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42, solver='liblinear'))])
